In [ ]:
# Project Setup:

- Initialize new Node.js project.
- Create necessary files (server.js, package.json)

# Directory tree

distributable_api/  
├── app/  
│   ├── __init__.py  
│   ├── main.py              # Core API logic  
│   ├── schemas.py           # Pydantic data models  
│   └── dependencies.py      # Auth & utilities  
├── tests/                   # Test scripts  
├── Dockerfile               # Containerization  
├── requirements.txt         # Dependencies  
└── README.md                # Usage/docs  

In [ ]:
from fastapi import FastAPI, HTTPException, Depends, status, Security
from fastapi.security import APIKeyHeader, OAuth2PasswordBearer
from pydantic import BaseModel, Field
from typing import Optional, List
import uuid
import datetime

In [ ]:
app = FastAPI(
    title="FastAPI",
    description="API development",
    version="2.1.0",
    openapi_tags=[
        {
            "name": "Authentication",
            "description": "Secure API key and token endpoints"
        },
        {
            "name": "Data Processing",
            "description": "Core business logic endpoints"
        },
        {
            "name": "System",
            "description": "Health checks and diagnostics"
        }
    ]
)


In [ ]:
# Adding security

api_key_header = APIKeyHeader(name="X-API-KEY", auto_error=False)
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/token")

VALID_API_KEYS = ["api_development"]
ACCESS_TOKENS = {"demo_token": "api_developer"}

def validate_api_key(api_key: str = Security(api_key_header)):
    if api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid API Key"
        )
    return api_key

def validate_token(token: str = Depends(oauth2_scheme)):
    if token not in ACCESS_TOKENS:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid access token"
        )
    return ACCESS_TOKENS[token]


In [ ]:
# Data Models
class TextAnalysisRequest(BaseModel):
    text: str = Field(..., example="FastAPI is amazing!", max_length=500)
    analysis_type: Optional[str] = Field(
        "basic",
        example="advanced",
        description="Analysis depth: 'basic' or 'advanced'"
    )

class AnalysisResult(BaseModel):
    request_id: str
    timestamp: datetime.datetime
    word_count: int
    character_count: int
    unique_words: int
    avg_word_length: float
    sentiment_score: Optional[float] = Field(
        None,
        description="Only available in advanced analysis"
    )
    processing_time_ms: float

class SystemHealth(BaseModel):
    status: str
    version: str
    uptime: float
    active_requests: int

In [ ]:
@app.post("/analyze-text",
          response_model=AnalysisResult,
          tags=["Data Processing"],
          summary="Text analysis endpoint",
          description="Data processing with configurable options")
async def analyze_text(
    request: TextAnalysisRequest,
    user: str = Depends(validate_token)
):
    """Text analysis with metrics and sentiment"""
    start_time = datetime.datetime.now()

    # Core processing logic
    words = request.text.split()
    word_count = len(words)
    char_count = len(request.text)
    unique_words = len(set(words))
    avg_length = sum(len(word) for word in words) / word_count if word_count > 0 else 0

    # Advanced analysis
    sentiment = None
    if request.analysis_type == "advanced":
        sentiment = calculate_sentiment(request.text)

    return {
        "request_id": str(uuid.uuid4()),
        "timestamp": datetime.datetime.now(),
        "word_count": word_count,
        "character_count": char_count,
        "unique_words": unique_words,
        "avg_word_length": round(avg_length, 2),
        "sentiment_score": sentiment,
        "processing_time_ms": round((datetime.datetime.now() - start_time).total_seconds() * 1000, 2)
    }

@app.get("/system/health",
         response_model=SystemHealth,
         tags=["System"],
         summary="System health check")
async def health_check(api_key: str = Depends(validate_api_key)):
    """API Development"""
    return {
        "status": "operational",
        "version": app.version,
        "uptime": 27.5,  # Hours
        "active_requests": 42
    }

@app.post("/token", tags=["Authentication"])
async def generate_token():
    """OAuth2 token endpoint (simulated)"""
    return {"access_token": "demo_token", "token_type": "bearer"}

In [ ]:
# Helper functions
def calculate_sentiment(text: str) -> float:
    """Mock sentiment analysis (replace with real NLP model)"""
    positive_words = ["good", "great", "excellent", "amazing", "love"]
    negative_words = ["bad", "terrible", "awful", "hate", "worst"]

    words = text.lower().split()
    positive = sum(1 for word in words if word in positive_words)
    negative = sum(1 for word in words if word in negative_words)

    total = len(words)
    if total == 0:
        return 0.0

    return round((positive - negative) / total * 10, 2)

In [ ]:
# Error handling
@app.exception_handler(HTTPException)
async def custom_http_exception_handler(request, exc):
    return JSONResponse(
        status_code=exc.status_code,
        content={
            "error": exc.detail,
            "request_id": request.headers.get("X-Request-ID", str(uuid.uuid4()))
        }
    )

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)